## Notebook — Sync Lucca → Supabase (BE Saisies de temps)

### Objectif
Alimenter la table Supabase `lucca_saisie_temps` (mirror) depuis l'API REST Lucca
Timesheet, pour permettre le suivi du temps déclaré par affaire BE.

### Source
API Lucca Timesheet (`https://<tenant>.ilucca.net/api/v3/timesheet/...`)
- Auth : `Authorization: Lucca application=<API_KEY>`
- Filtre : par plage de dates `date.since` / `date.until`

### Destination
- `lucca_saisie_temps` (Supabase) — clé de dédup `external_id` = id Lucca de la saisie
- Jointure côté SQL : `lucca_saisie_temps.code_site = be_affaires.code_affaire`
- Lookup `user_id` réalisé dans le notebook via map `id_lucca → profiles.id`
  (récupérée depuis Supabase au démarrage)

### Pattern d'envoi
Edge Function `bulk-upsert` (mode `upsert` total). Conflict key = `external_id`.
Idempotent — relançable, le delta s'additionne sans doublons.

### Configuration côté Fabric
Variables Base Parameters du notebook (à définir + injecter par le pipeline) :
- `SUPABASE_URL`, `SUPABASE_ANON_KEY`, `SYNC_SECRET` (déjà dispo, comme pour Divalto)
- `LUCCA_BASE_URL` (ex: `https://keon.ilucca.net`)
- `LUCCA_API_KEY`
- `SYNC_DAYS_BACK` (ex: `30` pour resync les 30 derniers jours)

### ⚠️ À adapter selon votre tenant Lucca
Le mapping des champs (notamment `imputations` → `code_site`) dépend du paramétrage
de l'axe analytique dans Lucca. Voir cellule 4 (mapping) — chercher les commentaires
`# TODO adapter` ou `# Vérifier`.

In [ ]:
# ─────────────────────────────────────────
# 0) Credentials & config
# ─────────────────────────────────────────

# env = notebookutils.variableLibrary.getLibrary("env-lucca")
# SUPABASE_URL       = env.SUPABASE_URL
# SYNC_SECRET        = env.SYNC_SECRET
# SUPABASE_ANON_KEY  = env.SUPABASE_PUBLISHABLE_KEY
# LUCCA_BASE_URL     = env.LUCCA_BASE_URL
# LUCCA_API_KEY      = env.LUCCA_API_KEY
# SYNC_DAYS_BACK     = int(env.SYNC_DAYS_BACK or 30)

import math, time, re, json, requests
from decimal import Decimal
from datetime import date, datetime, timedelta

BULK_UPSERT_URL = f"{SUPABASE_URL.rstrip('/')}/functions/v1/bulk-upsert"
BATCH_SIZE      = 500
SLEEP_S         = 0.02

# Plage de dates à synchroniser (rolling window)
TODAY     = date.today()
DATE_FROM = TODAY - timedelta(days=int(SYNC_DAYS_BACK))
DATE_TO   = TODAY

def supabase_headers(json_body: bool = True):
    h = {
        "Authorization":  f"Bearer {SUPABASE_ANON_KEY}",
        "apikey":         SUPABASE_ANON_KEY,
        "x-sync-secret":  SYNC_SECRET,
    }
    if json_body:
        h["Content-Type"] = "application/json"
    return h

def lucca_headers():
    return {
        "Authorization": f"Lucca application={LUCCA_API_KEY}",
        "Accept":        "application/json",
    }

print("[0] Config OK")
print(f"    SUPABASE_URL    = {SUPABASE_URL}")
print(f"    LUCCA_BASE_URL  = {LUCCA_BASE_URL}")
print(f"    Période sync    = {DATE_FROM} → {DATE_TO} ({SYNC_DAYS_BACK} jours)")

In [ ]:
# ─────────────────────────────────────────
# 1) Helpers serialisation JSON + envoi batch (identique pattern Divalto)
# ─────────────────────────────────────────

_CTRL_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

def _clean_str(s):
    if s is None:
        return None
    return _CTRL_RE.sub("", str(s)).rstrip() or None

def _to_jsonable(v):
    if v is None: return None
    if isinstance(v, bool): return v
    if isinstance(v, int): return v
    if isinstance(v, float):
        return None if (math.isnan(v) or math.isinf(v)) else round(v, 4)
    if isinstance(v, Decimal):
        try:
            f = float(v)
            return None if (math.isnan(f) or math.isinf(f)) else round(f, 4)
        except Exception:
            return _clean_str(str(v))
    if isinstance(v, (date, datetime)): return v.isoformat()
    if isinstance(v, str):
        s = _clean_str(v)
        if not s or s in ("1899-12-30", "0000-00-00"): return None
        return s
    if isinstance(v, dict): return {k: _to_jsonable(val) for k, val in v.items()}
    if isinstance(v, (list, tuple)): return [_to_jsonable(x) for x in v]
    return _clean_str(str(v))

def _sanitize(rec):
    return {k: _to_jsonable(v) for k, v in rec.items()}

def _post_batch(table: str, conflict_key: str, records: list, on_conflict: str = "upsert"):
    if not records:
        return 0, None
    payload = {
        "table":        table,
        "conflict_key": conflict_key,
        "records":      [_sanitize(r) for r in records],
        "on_conflict":  on_conflict,
    }
    try:
        r = requests.post(BULK_UPSERT_URL, headers=supabase_headers(), json=payload, timeout=180)
        if r.status_code in (200, 201, 204):
            return len(records), None
        return 0, f"HTTP {r.status_code}: {r.text[:300]}"
    except Exception as e:
        return 0, str(e)

def send_to_supabase(table: str, conflict_key: str, records: list, label: str):
    ok_total, err_total, err_msgs = 0, 0, []
    total = len(records)
    for i in range(0, total, BATCH_SIZE):
        chunk = records[i:i + BATCH_SIZE]
        nb_ok, err = _post_batch(table, conflict_key, chunk)
        ok_total += nb_ok
        if err:
            err_total += len(chunk)
            err_msgs.append(f"batch {i//BATCH_SIZE + 1}: {err}")
        time.sleep(SLEEP_S)
    print(f"[{label}] ✅ {ok_total}/{total} envoyés | ❌ {err_total} erreurs")
    for m in err_msgs:
        print(f"  ⚠️  {m}")
    return ok_total, err_total

print("[1] Helpers OK")

In [ ]:
# ─────────────────────────────────────────
# 2) Lookup map : id_lucca → profile_id (depuis Supabase)
# Permet de remplir lucca_saisie_temps.user_id dès l'insert
# ─────────────────────────────────────────

url = f"{SUPABASE_URL.rstrip('/')}/rest/v1/profiles"
params = {
    "select":   "id,id_lucca",
    "id_lucca": "not.is.null",
    "limit":    "5000",
}
r = requests.get(url, headers=supabase_headers(json_body=False), params=params, timeout=60)
r.raise_for_status()
profiles_rows = r.json()

ID_LUCCA_TO_PROFILE_ID = {}
for row in profiles_rows:
    if row.get("id_lucca"):
        try:
            key = int(str(row["id_lucca"]).strip())
            ID_LUCCA_TO_PROFILE_ID[key] = row["id"]
        except Exception:
            continue

print(f"[2] Lookup map : {len(ID_LUCCA_TO_PROFILE_ID)} profils Lucca -> Supabase")

In [ ]:
# ─────────────────────────────────────────
# 3) Fetch des saisies de temps Lucca (paginé)
# Endpoint : /api/v3/timesheet/legacytimesheet (à adapter selon ton tenant Lucca)
# Documentation : https://help.lucca.fr/api
# ─────────────────────────────────────────

TIMESHEET_URL = f"{LUCCA_BASE_URL.rstrip('/')}/api/v3/timesheet/legacytimesheet"
PAGE_SIZE = 1000

raw_entries = []
page = 0
while True:
    params = {
        "date.since": DATE_FROM.isoformat(),
        "date.until": DATE_TO.isoformat(),
        "paging":     f"{page * PAGE_SIZE},{PAGE_SIZE}",
        # TODO : adapter les champs (fields) selon ton API Lucca
        # Ex: 'fields': 'id,date,duration,owner.id,description,imputations.value'
    }
    r = requests.get(TIMESHEET_URL, headers=lucca_headers(), params=params, timeout=120)
    if r.status_code == 404:
        print(f"[3] ⚠️  404 sur {TIMESHEET_URL} - vérifier le path Lucca exact (timesheet vs timesheet/legacytimesheet)")
        break
    r.raise_for_status()
    body = r.json()
    items = body.get("data", {}).get("items") or body.get("items") or body.get("data") or []
    if not items:
        break
    raw_entries.extend(items)
    if len(items) < PAGE_SIZE:
        break
    page += 1
    time.sleep(0.05)

print(f"[3] Saisies Lucca récupérées : {len(raw_entries)}")
if raw_entries:
    print("    Exemple première entrée (clés disponibles) :")
    print("   ", list(raw_entries[0].keys()))

In [ ]:
# ─────────────────────────────────────────
# 4) Mapping Lucca → schéma lucca_saisie_temps
# ⚠️  Ce mapping DOIT être adapté selon le format réel renvoyé par votre tenant Lucca.
# Inspecter le 1er résultat (cellule 3) puis ajuster les .get() ci-dessous.
# ─────────────────────────────────────────

def extract_code_site(entry: dict) -> str | None:
    """
    Récupère le code_site (= code_affaire) depuis les imputations analytiques.
    Plusieurs formats possibles selon le paramétrage Lucca :
      - entry['imputations'] = [{'value': 'EVINZEPU', 'axis': 'CODE_SITE'}, ...]
      - entry['analyticAxis_1'] = 'EVINZEPU'
      - entry['cost_centers'] = [{'code': 'EVINZEPU'}]
    """
    # Cas 1 : liste imputations
    imps = entry.get("imputations") or []
    if isinstance(imps, list):
        for imp in imps:
            axis = (imp.get("axis") or imp.get("axisName") or "").upper()
            if axis in ("CODE_SITE", "AXE_0001", "AXE_0002", "AFFAIRE", "PROJET"):
                v = imp.get("value") or imp.get("code")
                if v:
                    return str(v).strip()
        # Fallback : 1re imputation non vide
        for imp in imps:
            v = imp.get("value") or imp.get("code")
            if v:
                return str(v).strip()

    # Cas 2 : champs plats
    for key in ("code_site", "codeSite", "analyticAxis_1", "axe_0001", "axe1"):
        v = entry.get(key)
        if v:
            return str(v).strip()

    return None

records = []
skipped_no_axis = 0
skipped_no_owner = 0
for entry in raw_entries:
    external_id = entry.get("id") or entry.get("Id")
    if external_id is None:
        continue

    # Owner (collaborateur Lucca)
    owner = entry.get("owner") or entry.get("ownerId") or entry.get("userId")
    if isinstance(owner, dict):
        id_lucca = owner.get("id")
    else:
        id_lucca = owner
    if id_lucca is None:
        skipped_no_owner += 1
        continue
    try:
        id_lucca_int = int(str(id_lucca).strip())
    except Exception:
        skipped_no_owner += 1
        continue

    # Code site = code_affaire (clé de jointure avec be_affaires)
    code_site = extract_code_site(entry)
    if not code_site:
        skipped_no_axis += 1
        continue

    # Durée : Lucca peut renvoyer en minutes (duration), heures (hours) ou ISO (PT2H30M)
    duree_minutes = entry.get("duration") or entry.get("durationInMinutes") or 0
    if isinstance(duree_minutes, (int, float)):
        duree_heures = round(float(duree_minutes) / 60.0, 2)
    else:
        duree_heures = 0.0

    # Date
    date_saisie = entry.get("date") or entry.get("startDate")
    if isinstance(date_saisie, str):
        date_saisie = date_saisie[:10]

    rec = {
        "external_id":        str(external_id),
        "id_lucca":           id_lucca_int,
        "user_id":            ID_LUCCA_TO_PROFILE_ID.get(id_lucca_int),
        "code_site":          code_site,
        "date_saisie":        date_saisie,
        "duree_heures":       duree_heures,
        "type_temps":         entry.get("type") or entry.get("timesheetType"),
        "libelle":            entry.get("description") or entry.get("comment"),
        "validated_by_lucca": bool(entry.get("isValidated") or entry.get("validated") or False),
        # raw conserve le payload Lucca brut pour debug si besoin
        "raw":                entry,
    }
    records.append(rec)

print(f"[4] Records prêts : {len(records)}")
print(f"    Ignorés (pas de code_site)  : {skipped_no_axis}")
print(f"    Ignorés (pas d'owner Lucca) : {skipped_no_owner}")

# Top 10 codes_site pour vérif rapide
from collections import Counter
by_code = Counter(r["code_site"] for r in records)
print("    Top 10 codes_site :")
for code, n in by_code.most_common(10):
    print(f"      {code} : {n} saisies")

In [ ]:
# ─────────────────────────────────────────
# 5) Envoi vers Supabase (lucca_saisie_temps)
# Conflict key = external_id → upsert idempotent
# ─────────────────────────────────────────

ok, err = send_to_supabase(
    table        = "lucca_saisie_temps",
    conflict_key = "external_id",
    records      = records,
    label        = "LUCCA_TEMPS",
)

In [ ]:
# ─────────────────────────────────────────
# 6) Résumé
# ─────────────────────────────────────────

users_sans_match = sum(1 for r in records if r['user_id'] is None)

print("")
print("=" * 60)
print("SYNC LUCCA TEMPS → SUPABASE — RÉSUMÉ")
print("=" * 60)
print(f"  Période                              : {DATE_FROM} → {DATE_TO}")
print(f"  Saisies récupérées Lucca             : {len(raw_entries)}")
print(f"  Records valides (avec code_site)     : {len(records)}")
print(f"  Upsertées Supabase                   : {ok}")
print(f"  Erreurs envoi                        : {err}")
print(f"  ⚠️  user_id non résolu (id_lucca)    : {users_sans_match}")
if err == 0:
    print("  ✅ Sync OK")
else:
    print("  ⚠️  Vérifier les logs ci-dessus")
print("=" * 60)